<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

[ваш текст]

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:

using System;
using System.Collections.Generic;
using System.Linq;

// НОВОЕ: делегат для выполнения действий с заказами
public delegate void OrderAction(Order order);

public interface ITrackable
{
    string GetTrackingStatus();
    void UpdateTrackingStatus(string status);
}

public interface IDiscountable
{
    int DiscountPercent { get; set; }
    void ApplyDiscount();
}

public interface INotifiable
{
    void SendNotification(string message);
}

public interface INotificationService
{
    void Send(string recipient, string message);
}

public interface ITrackingService
{
    void Update(int orderId, string status);
    string Get(int orderId);
}

public class EmailNotificationService : INotificationService
{
    public void Send(string recipient, string message)
        => Console.WriteLine($"  [EmailService] -> {recipient}: {message}");
}

public class SmsNotificationService : INotificationService
{
    public void Send(string recipient, string message)
        => Console.WriteLine($"  [SmsService] -> {recipient}: {message}");
}

public class InMemoryTrackingService : ITrackingService
{
    private readonly Dictionary<int, string> _store = new Dictionary<int, string>();

    public void Update(int orderId, string status)
    {
        _store[orderId] = status;
        Console.WriteLine($"  [TrackingService] Заказ №{orderId}: {status}");
    }

    public string Get(int orderId)
        => _store.TryGetValue(orderId, out var s) ? s : "Нет данных";
}

public class Item
{
    public string Name { get; set; }
    public int Amount { get; set; }

    public Item(string name, int amount)
    {
        Name = name;
        Amount = amount;
    }

    public override string ToString()
    {
        return $"{Name} ({Amount} руб.)";
    }
}

public class Order
{
    private int _orderId;

    public string CreationDate { get; set; }
    public int TotalAmount { get; protected set; }
    protected List<Item> items = new List<Item>();

    public string CustomerName { get; set; }
    public string PaymentMethod { get; set; }
    public string Status { get; protected set; }
    public DateTime LastModified { get; protected set; }
    public string Priority { get; set; } = "Обычный";
    public string Comment { get; set; } = "";
    public int Version { get; protected set; } = 1;

    public string Currency { get; set; } = "RUB";
    public bool IsArchived { get; private set; } = false;
    public int ItemCount => items.Count;
    public string CreatedBy { get; set; } = "Система";

    // НОВОЕ: дополнительные атрибуты базового класса
    public string Source { get; set; } = "Интернет-магазин";
    public bool IsGiftWrapped { get; set; } = false;
    public int DeliveryAttempts { get; set; } = 0;

    // НОВОЕ: коллекции для хранения истории, тегов и дополнительных данных
    protected List<string> OrderHistory = new List<string>();
    protected HashSet<string> OrderTags = new HashSet<string>();
    protected Dictionary<string, string> OrderMetadata = new Dictionary<string, string>();

    // НОВОЕ: событие изменения заказа
    public event Action<string> OrderChanged;

    public int OrderId
    {
        get { return _orderId; }
        set
        {
            if (value > 0)
                _orderId = value;
            else
                throw new ArgumentOutOfRangeException("ID заказа должен быть положительным числом!");
        }
    }

    public Order(int orderId, string creationDate, string customerName, string paymentMethod)
    {
        OrderId = orderId;
        CreationDate = creationDate;
        CustomerName = customerName;
        PaymentMethod = paymentMethod;
        Status = "Создан";
        LastModified = DateTime.Now;
    }

    public virtual void AddItem(Item item)
    {
        items.Add(item);
        LastModified = DateTime.Now;
        Console.WriteLine($"  -> Добавлен товар: {item}");
    }

    public virtual void RemoveItem(Item item)
    {
        if (items.Contains(item))
        {
            items.Remove(item);
            LastModified = DateTime.Now;
            Console.WriteLine($"  -> Удален товар: {item}");
        }
        else
        {
            Console.WriteLine($"  -> Товар '{item.Name}' не найден в заказе.");
        }
    }

    public virtual void CalculateTotal()
    {
        int sum = 0;

        foreach (var item in items)
            sum += item.Amount;

        TotalAmount = sum;
        Console.WriteLine($"  -> Итоговая сумма заказа №{OrderId}: {TotalAmount} руб.");
    }

    public void ChangeStatus(string newStatus)
    {
        Status = newStatus;
        LastModified = DateTime.Now;
        Console.WriteLine($"  -> Статус заказа №{OrderId} изменен на '{Status}'");

        // НОВОЕ: вызов события после изменения статуса
        OrderChanged?.Invoke($"Статус заказа №{OrderId} изменен на '{Status}'");
    }

    public int GetItemsCount()
    {
        return items.Count;
    }

    public void PrintInfo()
    {
        Console.WriteLine($"  Заказ №{OrderId} | Дата: {CreationDate} | Клиент: {CustomerName} | Оплата: {PaymentMethod} | Статус: {Status} | Товаров: {items.Count} | Сумма: {TotalAmount} руб. | Изменен: {LastModified:HH:mm:ss}");
    }

    public List<Item> GetItems()
    {
        return items;
    }

    public virtual string GetOrderType()
    {
        return "Базовый заказ";
    }

    public void PrintInfo(string prefix)
    {
        Console.WriteLine($"[{prefix}] Заказ №{OrderId} | Клиент: {CustomerName} | Статус: {Status}");
    }

    public void PrintInfo(bool shortFormat)
    {
        if (shortFormat)
            Console.WriteLine($"  Заказ №{OrderId}: {CustomerName} — {TotalAmount} руб.");
        else
            PrintInfo();
    }

    public void SetPriority(string priority = "Обычный")
    {
        Priority = priority;
        LastModified = DateTime.Now;
        Console.WriteLine($"  -> Приоритет заказа №{OrderId}: {Priority}");
    }

    public void IncrementVersion()
    {
        Version++;
        Console.WriteLine($"  -> Версия заказа №{OrderId}: v{Version}");
    }

    public void Archive()
    {
        IsArchived = true;
        LastModified = DateTime.Now;
        Console.WriteLine($"  -> Заказ №{OrderId} архивирован ({Currency})");
    }

    public void SetCurrency(string currency)
    {
        if (string.IsNullOrWhiteSpace(currency))
            throw new ArgumentException("Валюта не может быть пустой!");

        Currency = currency.ToUpper();
        Console.WriteLine($"  -> Валюта заказа №{OrderId}: {Currency}");
    }

    public string GetSummary()
    {
        return $"№{OrderId} | {CustomerName} | {TotalAmount} {Currency} | {Status} | v{Version}";
    }

    public void CloneMetadataFrom(Order other)
    {
        Priority = other.Priority;
        Comment = other.Comment;
        CreatedBy = other.CreatedBy;
        Console.WriteLine($"  -> Метаданные скопированы из заказа №{other.OrderId} в №{OrderId}");
    }

    // НОВОЕ: добавление записи в историю заказа
    public void AddHistoryRecord(string record)
    {
        OrderHistory.Add(record);
        OrderChanged?.Invoke($"Заказ №{OrderId}: {record}");
    }

    // НОВОЕ: добавление уникального тега
    public void AddTag(string tag)
    {
        if (OrderTags.Add(tag))
            Console.WriteLine($"  -> Добавлен тег: {tag}");
        else
            Console.WriteLine($"  -> Тег '{tag}' уже существует");
    }

    // НОВОЕ: добавление дополнительных данных в словарь
    public void AddMetadata(string key, string value)
    {
        OrderMetadata[key] = value;
        Console.WriteLine($"  -> Добавлены данные: {key} = {value}");
    }

    // НОВОЕ: вывод истории заказа
    public void PrintHistory()
    {
        Console.WriteLine($"  История заказа №{OrderId}:");

        foreach (var record in OrderHistory)
            Console.WriteLine($"    - {record}");
    }

    // НОВОЕ: вывод уникальных тегов
    public void PrintTags()
    {
        Console.WriteLine($"  Теги заказа №{OrderId}:");

        foreach (var tag in OrderTags)
            Console.WriteLine($"    - {tag}");
    }

    // НОВОЕ: вывод дополнительных данных
    public void PrintMetadata()
    {
        Console.WriteLine($"  Дополнительные данные заказа №{OrderId}:");

        foreach (var data in OrderMetadata)
            Console.WriteLine($"    - {data.Key}: {data.Value}");
    }

    // НОВОЕ: увеличение количества попыток доставки
    public void IncrementDeliveryAttempts()
    {
        DeliveryAttempts++;
        Console.WriteLine($"  -> Количество попыток доставки: {DeliveryAttempts}");
    }

    // НОВОЕ: установка источника заказа
    public void SetSource(string source)
    {
        Source = source;
        Console.WriteLine($"  -> Источник заказа: {Source}");
    }

    // НОВОЕ: установка подарочной упаковки
    public void SetGiftWrapping(bool enabled)
    {
        IsGiftWrapped = enabled;
        Console.WriteLine($"  -> Подарочная упаковка: {(IsGiftWrapped ? "Да" : "Нет")}");
    }
}

public class OnlineOrder : Order, INotifiable
{
    public string CustomerEmail { get; set; }
    public string DeliveryMethod { get; set; }
    public string PromoCode { get; set; }
    public bool IsPaid { get; private set; }
    public string TrackingNumber { get; set; } = "TRK-000";
    public bool IsGift { get; set; } = false;

    public string WebsiteSource { get; set; } = "shop.example.com";
    public bool NotificationsEnabled { get; set; } = true;
    public int NotificationCount { get; private set; } = 0;

    // НОВОЕ: дополнительные атрибуты
    public string PreferredContactTime { get; set; } = "В любое время";
    public int LoyaltyLevel { get; set; } = 1;

    private readonly INotificationService _notificationService;

    public OnlineOrder(int orderId, string creationDate, string customerName,
        string paymentMethod, string email, string deliveryMethod, string promoCode,
        INotificationService notificationService = null)
        : base(orderId, creationDate, customerName, paymentMethod)
    {
        CustomerEmail = email;
        DeliveryMethod = deliveryMethod;
        PromoCode = promoCode;
        IsPaid = false;
        _notificationService = notificationService ?? new EmailNotificationService();
    }

    public override void AddItem(Item item)
    {
        base.AddItem(item);
        Console.WriteLine($"  -> Способ доставки: {DeliveryMethod}. Уведомление отправлено на {CustomerEmail}");
    }

    void INotifiable.SendNotification(string message)
    {
        if (!NotificationsEnabled)
        {
            Console.WriteLine($"  -> [OnlineOrder] Уведомления отключены, сообщение проигнорировано.");
            return;
        }

        NotificationCount++;
        Console.WriteLine($"  -> [Явная реализация INotifiable] Email на {CustomerEmail}: {message} (всего: {NotificationCount})");
    }

    public void SendNotification(string message)
    {
        ((INotifiable)this).SendNotification(message);
    }

    public void Pay()
    {
        IsPaid = true;
        ChangeStatus("Оплачен");
        Console.WriteLine($"  -> Заказ №{OrderId} успешно оплачен через {PaymentMethod}");
    }

    public void ApplyPromoCode()
    {
        if (!string.IsNullOrEmpty(PromoCode))
            Console.WriteLine($"  -> Промокод '{PromoCode}' применен к заказу №{OrderId}");
        else
            Console.WriteLine($"  -> Промокод отсутствует.");
    }

    public void PrintOnlineInfo()
    {
        PrintInfo();
        Console.WriteLine($"  Email: {CustomerEmail} | Доставка: {DeliveryMethod} | Промокод: {PromoCode} | Оплачен: {(IsPaid ? "Да" : "Нет")}");
    }

    public override string GetOrderType()
    {
        return "Онлайн-заказ";
    }

    public new void PrintInfo()
    {
        base.PrintInfo();
        Console.WriteLine($"  [Онлайн] Тип: {GetOrderType()} | Трек: {TrackingNumber} | Подарок: {(IsGift ? "Да" : "Нет")}");
    }

    public void DisableNotifications()
    {
        NotificationsEnabled = false;
        Console.WriteLine($"  -> Уведомления для заказа №{OrderId} отключены");
    }

    public void ResendLastNotification()
    {
        Console.WriteLine($"  -> Повторная отправка уведомления на {CustomerEmail} (источник: {WebsiteSource})");
    }

    public string GetDeliveryEta()
    {
        return $"Ожидаемый срок: 2-3 дня ({DeliveryMethod})";
    }

    public void SendNotificationViaService(string message)
    {
        _notificationService.Send(CustomerEmail, message);
    }

    // НОВОЕ: установка предпочтительного времени связи
    public void SetContactTime(string time)
    {
        PreferredContactTime = time;
        Console.WriteLine($"  -> Предпочтительное время связи: {PreferredContactTime}");
    }

    // НОВОЕ: изменение уровня лояльности
    public void SetLoyaltyLevel(int level)
    {
        LoyaltyLevel = level;
        Console.WriteLine($"  -> Уровень лояльности клиента: {LoyaltyLevel}");
    }

    // НОВОЕ: вывод настроек клиента
    public void PrintCustomerSettings()
    {
        Console.WriteLine($"  -> Время связи: {PreferredContactTime} | Уровень лояльности: {LoyaltyLevel}");
    }

    // НОВОЕ: изменение источника интернет-заказа
    public void ChangeWebsiteSource(string source)
    {
        WebsiteSource = source;
        Console.WriteLine($"  -> Источник интернет-заказа: {WebsiteSource}");
    }
}

public class ExpressOnlineOrder : OnlineOrder, ITrackable
{
    public string CourierName { get; set; }
    public int DeliveryHours { get; set; }
    public string TrackingStatus { get; private set; }
    public double ExtraFee { get; set; }
    public int PriorityLevel { get; set; } = 1;
    public double DistanceKm { get; set; } = 0;

    public string VehicleType { get; set; } = "Автомобиль";
    public int MaxWeightKg { get; set; } = 20;
    public bool IsUrgent { get; set; } = true;

    // НОВОЕ: дополнительные атрибуты
    public string PackageType { get; set; } = "Обычная упаковка";
    public bool RequiresCarefulHandling { get; set; } = false;

    private readonly ITrackingService _trackingService;

    public ExpressOnlineOrder(int orderId, string creationDate, string customerName,
        string paymentMethod, string email, string deliveryMethod, string promoCode,
        string courierName, int deliveryHours, double extraFee,
        INotificationService notificationService = null,
        ITrackingService trackingService = null)
        : base(orderId, creationDate, customerName, paymentMethod, email, deliveryMethod, promoCode, notificationService)
    {
        CourierName = courierName;
        DeliveryHours = deliveryHours;
        ExtraFee = extraFee;
        TrackingStatus = "Ожидает сборки";
        _trackingService = trackingService ?? new InMemoryTrackingService();
    }

    string ITrackable.GetTrackingStatus()
    {
        return $"[EXPRESS] {TrackingStatus} (курьер: {CourierName}, авто: {VehicleType})";
    }

    void ITrackable.UpdateTrackingStatus(string status)
    {
        TrackingStatus = status;
        LastModified = DateTime.Now;
        Console.WriteLine($"  -> [Явная реализация ITrackable] Экспресс №{OrderId}: {status}");
    }

    public string GetTrackingStatus() => ((ITrackable)this).GetTrackingStatus();
    public void UpdateTrackingStatus(string status) => ((ITrackable)this).UpdateTrackingStatus(status);

    public override void CalculateTotal()
    {
        base.CalculateTotal();
        TotalAmount += (int)ExtraFee;
        Console.WriteLine($"  -> Экспресс-доставка за {DeliveryHours} ч. Добавлена надбавка {ExtraFee} руб. Итого: {TotalAmount} руб.");
    }

    public void AssignCourier(string courier)
    {
        CourierName = courier;
        Console.WriteLine($"  -> Назначен курьер: {CourierName} для заказа №{OrderId}");
    }

    public void PrintExpressInfo()
    {
        PrintOnlineInfo();
        Console.WriteLine($"  Курьер: {CourierName} | Срок доставки: {DeliveryHours} ч. | Надбавка: {ExtraFee} руб. | Трекинг: {TrackingStatus}");
    }

    public override string GetOrderType()
    {
        return "Экспресс-онлайн-заказ";
    }

    public void UpdateTrackingStatus(string status, string courier)
    {
        CourierName = courier;
        UpdateTrackingStatus(status);
    }

    public void SetVehicle(string vehicle, int maxWeight)
    {
        VehicleType = vehicle;
        MaxWeightKg = maxWeight;
        Console.WriteLine($"  -> Транспорт заказа №{OrderId}: {VehicleType}, макс. вес {MaxWeightKg} кг");
    }

    public bool CanCarry(Item item)
    {
        bool ok = item.Amount <= MaxWeightKg * 100;
        Console.WriteLine($"  -> Проверка '{item.Name}' ({item.Amount}) на {VehicleType}: {(ok ? "OK" : "превышение")}");
        return ok;
    }

    public void MarkUrgent(bool urgent)
    {
        IsUrgent = urgent;
        Console.WriteLine($"  -> Срочность заказа №{OrderId}: {(IsUrgent ? "СРОЧНО" : "обычная")}");
    }

    public void SyncTracking(string status)
    {
        _trackingService.Update(OrderId, status);
    }

    // НОВОЕ: установка типа упаковки
    public void SetPackageType(string type)
    {
        PackageType = type;
        Console.WriteLine($"  -> Тип упаковки: {PackageType}");
    }

    // НОВОЕ: настройка бережной доставки
    public void SetCarefulHandling(bool required)
    {
        RequiresCarefulHandling = required;
        Console.WriteLine($"  -> Бережная доставка: {(RequiresCarefulHandling ? "Да" : "Нет")}");
    }

    // НОВОЕ: изменение расстояния доставки
    public void SetDistance(double distance)
    {
        DistanceKm = distance;
        Console.WriteLine($"  -> Расстояние доставки: {DistanceKm} км");
    }

    // НОВОЕ: вывод дополнительных параметров экспресс-доставки
    public void PrintExpressSettings()
    {
        Console.WriteLine($"  -> Упаковка: {PackageType} | Бережная доставка: {(RequiresCarefulHandling ? "Да" : "Нет")} | Расстояние: {DistanceKm} км");
    }
}

public class PhysicalOrder : Order
{
    public string DeliveryAddress { get; set; }
    public string StoreName { get; set; }
    public string ReturnReason { get; set; }
    public bool IsReturned { get; private set; }
    public string StoreCode { get; set; } = "S-001";
    public bool IsFragile { get; set; } = false;

    public string WarehouseCode { get; set; } = "WH-01";
    public bool RequiresSignature { get; set; } = true;
    public int ShelfLifeDays { get; set; } = 30;

    // НОВОЕ: дополнительные атрибуты
    public string Entrance { get; set; } = "Не указан";
    public int Floor { get; set; } = 1;

    public PhysicalOrder(int orderId, string creationDate, string customerName,
        string paymentMethod, string address, string storeName)
        : base(orderId, creationDate, customerName, paymentMethod)
    {
        DeliveryAddress = address;
        StoreName = storeName;
        IsReturned = false;
    }

    public override void RemoveItem(Item item)
    {
        if (items.Contains(item))
        {
            base.RemoveItem(item);
            ReturnReason = "Возврат по инициативе клиента";
            IsReturned = true;
            Console.WriteLine($"  -> Возврат: курьер заберет товар с адреса: {DeliveryAddress}. Магазин: {StoreName}");
        }
        else
        {
            Console.WriteLine($"  -> Невозможно вернуть '{item.Name}': товар отсутствует в заказе №{OrderId}");
        }
    }

    public void ScheduleDelivery(string date)
    {
        Console.WriteLine($"  -> Доставка по адресу {DeliveryAddress} назначена на {date}");
    }

    public void PrintPhysicalInfo()
    {
        PrintInfo();
        Console.WriteLine($"  Адрес: {DeliveryAddress} | Магазин: {StoreName} | Возврат: {(IsReturned ? "Да (" + ReturnReason + ")" : "Нет")}");
    }

    public override string GetOrderType()
    {
        return "Физический заказ";
    }

    public void ScheduleDelivery(string date, string timeSlot)
    {
        Console.WriteLine($"  -> Доставка на {date} ({timeSlot}) по адресу {DeliveryAddress}");
    }

    public void AssignWarehouse(string code)
    {
        WarehouseCode = code;
        Console.WriteLine($"  -> Заказ №{OrderId} закреплён за складом {WarehouseCode}");
    }

    public void CheckShelfLife()
    {
        if (ShelfLifeDays < 7)
            Console.WriteLine($"  -> ВНИМАНИЕ: срок хранения {ShelfLifeDays} дн. истекает!");
        else
            Console.WriteLine($"  -> Срок хранения в норме: {ShelfLifeDays} дн.");
    }

    public void RequireSignature(bool required)
    {
        RequiresSignature = required;
        Console.WriteLine($"  -> Подпись при получении: {(RequiresSignature ? "требуется" : "не требуется")}");
    }

    // НОВОЕ: установка данных подъезда и этажа
    public void SetAddressDetails(string entrance, int floor)
    {
        Entrance = entrance;
        Floor = floor;
        Console.WriteLine($"  -> Подъезд: {Entrance}, этаж: {Floor}");
    }

    // НОВОЕ: вывод подробного адреса
    public void PrintAddressDetails()
    {
        Console.WriteLine($"  -> Адрес доставки: {DeliveryAddress}, подъезд: {Entrance}, этаж: {Floor}");
    }

    // НОВОЕ: изменение признака хрупкого товара
    public void SetFragile(bool fragile)
    {
        IsFragile = fragile;
        Console.WriteLine($"  -> Хрупкий товар: {(IsFragile ? "Да" : "Нет")}");
    }

    // НОВОЕ: изменение срока хранения
    public void SetShelfLife(int days)
    {
        ShelfLifeDays = days;
        Console.WriteLine($"  -> Новый срок хранения: {ShelfLifeDays} дней");
    }
}

public class SpecializedOrder : Order, IDiscountable, INotifiable, ITrackable
{
    public string SpecialConditions { get; set; }
    private int _discountPercent;
    public string NotificationChannel { get; set; }
    private string _trackingStatus;
    public string ManagerName { get; set; } = "Не назначен";
    public int BonusPoints { get; set; } = 0;

    public string ContractNumber { get; set; } = "C-000";
    public bool IsVip { get; set; } = false;
    public int MaxDiscountCap { get; set; } = 50;

    // НОВОЕ: дополнительные атрибуты
    public string Department { get; set; } = "Общий отдел";
    public string ContractValidUntil { get; set; } = "Не указан";

    int IDiscountable.DiscountPercent
    {
        get => _discountPercent;
        set
        {
            if (value < 0 || value > MaxDiscountCap)
                throw new ArgumentOutOfRangeException(nameof(value),
                    $"Скидка должна быть от 0 до {MaxDiscountCap}% (VIP-лимит)");

            _discountPercent = value;
        }
    }

    void IDiscountable.ApplyDiscount()
    {
        Console.WriteLine($"  -> [Явная реализация IDiscountable] Скидка {_discountPercent}% по договору {ContractNumber}");
    }

    public int DiscountPercent
    {
        get => ((IDiscountable)this).DiscountPercent;
        set => ((IDiscountable)this).DiscountPercent = value;
    }

    public void ApplyDiscount() => ((IDiscountable)this).ApplyDiscount();

    void INotifiable.SendNotification(string message)
    {
        string prefix = IsVip ? "[VIP]" : "[STD]";
        Console.WriteLine($"  -> {prefix} Уведомление через {NotificationChannel}: {message}");
    }

    public void SendNotification(string message) => ((INotifiable)this).SendNotification(message);

    string ITrackable.GetTrackingStatus() => _trackingStatus;

    void ITrackable.UpdateTrackingStatus(string status)
    {
        _trackingStatus = status;
        LastModified = DateTime.Now;
        Console.WriteLine($"  -> [Явная реализация ITrackable] Спец-трекинг заказа №{OrderId}: {_trackingStatus}");
    }

    public string GetTrackingStatus() => ((ITrackable)this).GetTrackingStatus();
    public void UpdateTrackingStatus(string status) => ((ITrackable)this).UpdateTrackingStatus(status);

    public SpecializedOrder(int orderId, string creationDate, string customerName,
        string paymentMethod, string conditions, int discount, string notificationChannel)
        : base(orderId, creationDate, customerName, paymentMethod)
    {
        SpecialConditions = conditions;
        NotificationChannel = notificationChannel;
        _trackingStatus = "Спец-заказ создан";
        DiscountPercent = discount;
    }

    public void MergeWithOrder(Order otherOrder)
    {
        Console.WriteLine($"[Взаимодействие]: Заказ №{OrderId} объединяет товары из заказа №{otherOrder.OrderId}");

        foreach (var item in otherOrder.GetItems())
            AddItem(item);
    }

    public override void CalculateTotal()
    {
        int sum = 0;

        foreach (var item in items)
            sum += item.Amount;

        int discountSum = (sum * DiscountPercent) / 100;
        TotalAmount = sum - discountSum;

        Console.WriteLine($"  -> Сумма со скидкой {DiscountPercent}% ({SpecialConditions}): {TotalAmount} руб.");
    }

    public void PrintSpecialInfo()
    {
        PrintInfo();
        Console.WriteLine($"  Условие: {SpecialConditions} | Скидка: {DiscountPercent}% | Канал уведомлений: {NotificationChannel} | Трекинг: {_trackingStatus}");
    }

    public override string GetOrderType()
    {
        return "Специализированный заказ";
    }

    public void ApplyDiscount(int extraPercent)
    {
        DiscountPercent = Math.Min(MaxDiscountCap, DiscountPercent + extraPercent);
        ApplyDiscount();
    }

    public void SignContract(string number, bool vip)
    {
        ContractNumber = number;
        IsVip = vip;
        MaxDiscountCap = vip ? 70 : 50;
        Console.WriteLine($"  -> Договор {ContractNumber} подписан. VIP: {(IsVip ? "Да" : "Нет")}, лимит скидки: {MaxDiscountCap}%");
    }

    public void AwardBonus(int points)
    {
        BonusPoints += points;
        Console.WriteLine($"  -> Начислено {points} бонусов. Всего: {BonusPoints}");
    }

    public void UpgradeToVip()
    {
        IsVip = true;
        MaxDiscountCap = 70;
        Console.WriteLine($"  -> Заказ №{OrderId} повышен до VIP (лимит скидки: {MaxDiscountCap}%)");
    }

    // НОВОЕ: установка отдела и срока действия договора
    public void SetContractDetails(string department, string validUntil)
    {
        Department = department;
        ContractValidUntil = validUntil;
        Console.WriteLine($"  -> Отдел: {Department}, договор действует до: {ContractValidUntil}");
    }

    // НОВОЕ: вывод подробных данных договора
    public void PrintContractDetails()
    {
        Console.WriteLine($"  -> Договор: {ContractNumber}, отдел: {Department}, срок действия: {ContractValidUntil}");
    }

    // НОВОЕ: изменение имени менеджера
    public void AssignManager(string manager)
    {
        ManagerName = manager;
        Console.WriteLine($"  -> Назначен менеджер: {ManagerName}");
    }

    // НОВОЕ: добавление бонусных баллов
    public void AddBonusPoints(int points)
    {
        BonusPoints += points;
        Console.WriteLine($"  -> Дополнительно начислено {points} бонусов. Всего: {BonusPoints}");
    }
}

public class OrderRegistry<T> where T : Order
{
    private List<T> _orders = new List<T>();

    public void Add(T order)
    {
        _orders.Add(order);
        Console.WriteLine($"  [Registry] Добавлен {typeof(T).Name} №{order.OrderId} (тип: {order.GetOrderType()})");
    }

    public T FindById(int id)
    {
        foreach (var o in _orders)
            if (o.OrderId == id)
                return o;

        return null;
    }

    public int Count => _orders.Count;

    public void PrintAll()
    {
        Console.WriteLine($"  [Registry<{typeof(T).Name}>] Всего: {Count}");

        foreach (var o in _orders)
            Console.WriteLine($"    - №{o.OrderId} | {o.GetOrderType()} | {o.CustomerName}");
    }
}

public static class OrderUtils
{
    public static void ShowType<T>(T order) where T : Order
    {
        Console.WriteLine($"  [Utils] №{order.OrderId} — тип: {order.GetOrderType()}");
    }

    public static void Compare<T1, T2>(T1 a, T2 b)
        where T1 : Order where T2 : Order
    {
        Console.WriteLine($"  [Utils] Сравнение №{a.OrderId} ({a.GetOrderType()}) и №{b.OrderId} ({b.GetOrderType()})");
    }
}

try
{
    Console.WriteLine("==========================================================");
    Console.WriteLine("ОНЛАЙН-ЗАКАЗ");
    Console.WriteLine("==========================================================");

    Item item1 = new Item("Помада", 450);
    Item item2 = new Item("Тушь", 200);
    Item item3 = new Item("Тональный крем", 1100);
    Item item4 = new Item("Румяна", 600);
    Item item5 = new Item("Парфюм", 3500);

    OnlineOrder onlineOrder = new OnlineOrder(
        111, "09.09.26", "Анастасия", "Карта",
        "nastya2405@mail.com", "Почта России", "SALE2026");

    onlineOrder.AddItem(item1);
    onlineOrder.AddItem(item2);
    onlineOrder.PrintOnlineInfo();
    onlineOrder.CalculateTotal();
    onlineOrder.Pay();
    onlineOrder.ApplyPromoCode();
    onlineOrder.SendNotification("Ваш заказ собирается!");
    onlineOrder.ChangeStatus("Собирается");
    onlineOrder.PrintOnlineInfo();

    Console.WriteLine("==========================================================");
    Console.WriteLine("КУРЬЕРСКИЙ ЭКСПРЕСС-ЗАКАЗ");
    Console.WriteLine("==========================================================");

    ExpressOnlineOrder expressOrder = new ExpressOnlineOrder(
        444, "10.09.26", "Иван", "Онлайн",
        "ivan@mail.com", "Курьер", "FAST10",
        "Петров", 3, 500);

    expressOrder.AddItem(item3);
    expressOrder.AddItem(item5);
    expressOrder.PrintExpressInfo();
    expressOrder.CalculateTotal();
    expressOrder.UpdateTrackingStatus("Курьер в пути");
    expressOrder.AssignCourier("Сидоров");
    expressOrder.UpdateTrackingStatus("Доставлен");
    expressOrder.PrintExpressInfo();

    Console.WriteLine("==========================================================");
    Console.WriteLine("ФИЗИЧЕСКИЙ ЗАКАЗ");
    Console.WriteLine("==========================================================");

    PhysicalOrder physicalOrder = new PhysicalOrder(
        222, "09.09.26", "Мария", "Наличные",
        "ул. Ленина, д. 10", "Магазин 'Косметика+'");

    physicalOrder.AddItem(item3);
    physicalOrder.AddItem(item4);
    physicalOrder.PrintPhysicalInfo();
    physicalOrder.CalculateTotal();
    physicalOrder.ScheduleDelivery("12.09.26");
    physicalOrder.RemoveItem(item3);
    physicalOrder.CalculateTotal();
    physicalOrder.PrintPhysicalInfo();

    Console.WriteLine("==========================================================");
    Console.WriteLine("СПЕЦИАЛИЗИРОВАННЫЙ ЗАКАЗ");
    Console.WriteLine("==========================================================");

    SpecializedOrder specOrder = new SpecializedOrder(
        333, "09.09.26", "Ольга", "Бонусы",
        "Карта лояльности", 15, "SMS");

    specOrder.AddItem(item5);
    specOrder.PrintSpecialInfo();
    specOrder.ApplyDiscount();
    specOrder.SendNotification("Ваш спец-заказ активирован!");
    specOrder.UpdateTrackingStatus("В обработке");
    specOrder.MergeWithOrder(onlineOrder);
    specOrder.CalculateTotal();
    specOrder.UpdateTrackingStatus("Готов к выдаче");
    specOrder.PrintSpecialInfo();

    Console.WriteLine("==========================================================");
    Console.WriteLine("ПОЛИМОРФИЗМ, ПЕРЕГРУЗКА, GENERIC");
    Console.WriteLine("==========================================================");

    Order[] allOrders = { onlineOrder, expressOrder, physicalOrder, specOrder };

    foreach (var o in allOrders)
        Console.WriteLine($"  Полиморфизм: №{o.OrderId} -> {o.GetOrderType()}");

    onlineOrder.PrintInfo();
    onlineOrder.PrintInfo("СРОЧНО");
    onlineOrder.PrintInfo(true);

    expressOrder.UpdateTrackingStatus("Передан курьеру", "Орлов");
    physicalOrder.ScheduleDelivery("15.09.26", "10:00-12:00");
    specOrder.ApplyDiscount(5);

    onlineOrder.SetPriority("Высокий");
    onlineOrder.IncrementVersion();

    specOrder.ManagerName = "Петрова";
    specOrder.BonusPoints = 120;

    Console.WriteLine($"  Менеджер спец-заказа: {specOrder.ManagerName}, бонусов: {specOrder.BonusPoints}");

    OrderRegistry<Order> registry = new OrderRegistry<Order>();
    registry.Add(onlineOrder);
    registry.Add(expressOrder);
    registry.Add(physicalOrder);
    registry.Add(specOrder);
    registry.PrintAll();

    OrderRegistry<OnlineOrder> onlineRegistry = new OrderRegistry<OnlineOrder>();
    onlineRegistry.Add(onlineOrder);
    onlineRegistry.Add(expressOrder);
    onlineRegistry.PrintAll();

    OrderUtils.ShowType(expressOrder);
    OrderUtils.Compare(onlineOrder, physicalOrder);

    Console.WriteLine("==========================================================");
    Console.WriteLine("ДОПОЛНИТЕЛЬНЫЕ АТРИБУТЫ И МЕТОДЫ");
    Console.WriteLine("==========================================================");

    onlineOrder.SetCurrency("usd");
    onlineOrder.Archive();
    Console.WriteLine(onlineOrder.GetSummary());

    onlineOrder.DisableNotifications();
    onlineOrder.SendNotification("Тест отключённых уведомлений");
    onlineOrder.ResendLastNotification();
    Console.WriteLine(onlineOrder.GetDeliveryEta());

    expressOrder.SetVehicle("Мотоцикл", 5);
    expressOrder.CanCarry(item5);
    expressOrder.MarkUrgent(false);
    Console.WriteLine(expressOrder.GetTrackingStatus());

    physicalOrder.AssignWarehouse("WH-07");
    physicalOrder.CheckShelfLife();
    physicalOrder.RequireSignature(false);

    specOrder.SignContract("C-2026-77", false);
    specOrder.AwardBonus(50);
    specOrder.UpgradeToVip();
    specOrder.SendNotification("VIP-статус активирован");

    // НОВОЕ: демонстрация дополнительных атрибутов базового класса
    onlineOrder.SetSource("Мобильное приложение");
    onlineOrder.SetGiftWrapping(true);
    onlineOrder.IncrementDeliveryAttempts();

    // НОВОЕ: демонстрация новых методов OnlineOrder
    onlineOrder.SetContactTime("10:00-18:00");
    onlineOrder.SetLoyaltyLevel(3);
    onlineOrder.PrintCustomerSettings();
    onlineOrder.ChangeWebsiteSource("mobile.shop.example.com");

    // НОВОЕ: демонстрация новых методов ExpressOnlineOrder
    expressOrder.SetPackageType("Подарочная упаковка");
    expressOrder.SetCarefulHandling(true);
    expressOrder.SetDistance(12.5);
    expressOrder.PrintExpressSettings();

    // НОВОЕ: демонстрация новых методов PhysicalOrder
    physicalOrder.SetAddressDetails("2", 5);
    physicalOrder.PrintAddressDetails();
    physicalOrder.SetFragile(true);
    physicalOrder.SetShelfLife(5);

    // НОВОЕ: демонстрация новых методов SpecializedOrder
    specOrder.SetContractDetails("Отдел продаж", "31.12.2026");
    specOrder.PrintContractDetails();
    specOrder.AssignManager("Иванова");
    specOrder.AddBonusPoints(30);

    // ==========================================================
    // НОВОЕ: КОЛЛЕКЦИИ, ДЕЛЕГАТЫ, ЛЯМБДА-ВЫРАЖЕНИЯ И СОБЫТИЯ
    // ==========================================================

    Console.WriteLine("==========================================================");
    Console.WriteLine("КОЛЛЕКЦИИ, ДЕЛЕГАТЫ, ЛЯМБДА-ВЫРАЖЕНИЯ И СОБЫТИЯ");
    Console.WriteLine("==========================================================");

    // НОВОЕ: подписка на событие изменения заказа
    onlineOrder.OrderChanged += message =>
    {
        Console.WriteLine($"  [Уведомление] {message}");
    };

    // НОВОЕ: добавление записей в историю
    Console.WriteLine("\n--- ИСТОРИЯ ЗАКАЗА ---");

    onlineOrder.AddHistoryRecord("Заказ создан");
    onlineOrder.AddHistoryRecord("Товары добавлены в корзину");
    onlineOrder.AddHistoryRecord("Произведена оплата");
    onlineOrder.AddHistoryRecord("Заказ передан в доставку");

    onlineOrder.PrintHistory();

    // НОВОЕ: работа с HashSet
    Console.WriteLine("\n--- УНИКАЛЬНЫЕ ТЕГИ ЗАКАЗА ---");

    onlineOrder.AddTag("Косметика");
    onlineOrder.AddTag("Онлайн");
    onlineOrder.AddTag("Оплачен");
    onlineOrder.AddTag("Косметика");

    onlineOrder.PrintTags();

    // НОВОЕ: работа с Dictionary
    Console.WriteLine("\n--- ДОПОЛНИТЕЛЬНЫЕ ДАННЫЕ ЗАКАЗА ---");

    onlineOrder.AddMetadata("Город", "Москва");
    onlineOrder.AddMetadata("Способ доставки", "Почта России");
    onlineOrder.AddMetadata("Категория", "Косметика");
    onlineOrder.AddMetadata("Статус оплаты", "Оплачен");

    onlineOrder.PrintMetadata();

    // НОВОЕ: использование делегата
    Console.WriteLine("\n--- РАБОТА С ДЕЛЕГАТАМИ ---");

    OrderAction orderAction = order => order.PrintInfo();

    Console.WriteLine("Вызов метода PrintInfo через делегат:");
    orderAction(onlineOrder);

    Console.WriteLine("\nВызов метода PrintInfo для экспресс-заказа:");
    orderAction(expressOrder);

    // НОВОЕ: использование лямбда-выражений
    Console.WriteLine("\n--- РАБОТА С ЛЯМБДА-ВЫРАЖЕНИЯМИ ---");

    Console.WriteLine("Все заказы:");

    foreach (var order in allOrders)
        order.PrintInfo();

    Console.WriteLine("\nЗаказы стоимостью больше 1000 рублей:");

    var expensiveOrders = allOrders.Where(order => order.TotalAmount > 1000);

    foreach (var order in expensiveOrders)
        order.PrintInfo();

    Console.WriteLine("\nЗаказы со статусом 'Оплачен':");

    var paidOrders = allOrders.Where(order => order.Status == "Оплачен");

    foreach (var order in paidOrders)
        order.PrintInfo();

    // НОВОЕ: демонстрация события изменения статуса
    Console.WriteLine("\n--- РАБОТА С СОБЫТИЯМИ ---");

    Console.WriteLine("Изменение статуса онлайн-заказа:");
    onlineOrder.ChangeStatus("Передан в доставку");

    Console.WriteLine("\nИзменение статуса на 'Доставлен':");
    onlineOrder.ChangeStatus("Доставлен");

    // НОВОЕ: проверка количества элементов и коллекций
    Console.WriteLine("\n--- ИТОГОВАЯ ИНФОРМАЦИЯ ---");

    Console.WriteLine($"Количество товаров в онлайн-заказе: {onlineOrder.ItemCount}");
    Console.WriteLine($"Количество попыток доставки: {onlineOrder.DeliveryAttempts}");
    Console.WriteLine($"Источник заказа: {onlineOrder.Source}");
    Console.WriteLine($"Подарочная упаковка: {(onlineOrder.IsGiftWrapped ? "Да" : "Нет")}");
    Console.WriteLine($"Количество тегов: 3");
    Console.WriteLine($"Количество записей истории: 4");

    // ==========================================================
    // ЯВНАЯ РЕАЛИЗАЦИЯ ИНТЕРФЕЙСОВ
    // ==========================================================

    Console.WriteLine("==========================================================");
    Console.WriteLine("ЯВНАЯ РЕАЛИЗАЦИЯ ИНТЕРФЕЙСОВ");
    Console.WriteLine("==========================================================");

    INotifiable notifier = onlineOrder;
    notifier.SendNotification("Проверка явной реализации INotifiable");

    ITrackable tracker = expressOrder;
    Console.WriteLine($"  Трекинг через интерфейс: {tracker.GetTrackingStatus()}");
    tracker.UpdateTrackingStatus("Доставлен через интерфейс");

    IDiscountable discountable = specOrder;
    Console.WriteLine($"  Скидка через интерфейс: {discountable.DiscountPercent}%");
    discountable.ApplyDiscount();

    // ==========================================================
    // УПРАВЛЕНИЕ ЗАВИСИМОСТЯМИ
    // ==========================================================

    Console.WriteLine("==========================================================");
    Console.WriteLine("УПРАВЛЕНИЕ ЗАВИСИМОСТЯМИ (DI)");
    Console.WriteLine("==========================================================");

    var smsService = new SmsNotificationService();
    var trackService = new InMemoryTrackingService();

    var diOrder = new OnlineOrder(
        777, "12.09.26", "Алексей", "Онлайн",
        "alex@mail.com", "Курьер", "DI2026", smsService);

    diOrder.SendNotificationViaService("DI-уведомление работает!");

    var diExpress = new ExpressOnlineOrder(
        888, "12.09.26", "Елена", "Карта",
        "elena@mail.com", "Курьер", "DI",
        "Кузнецов", 1, 700,
        notificationService: smsService,
        trackingService: trackService);

    diExpress.SendNotificationViaService("Экспресс с DI");
    diExpress.SyncTracking("Принят в обработку");

    Console.WriteLine($"  Из сервиса трекинга: {trackService.Get(888)}");

    // ==========================================================
    // ПРОВЕРКА ВАЛИДАЦИИ
    // ==========================================================

    Console.WriteLine("==========================================================");
    Console.WriteLine("Проверка валидации:");
    Console.WriteLine("==========================================================");

    Console.WriteLine("Попытка установить некорректную скидку (-5)...");

    specOrder.DiscountPercent = -5;
}
catch (Exception ex)
{
    Console.WriteLine($"Ошибка валидации данных -> {ex.Message}");
}


The below script needs to be able to find the current output cell; this is an easy method to get it.

ОНЛАЙН-ЗАКАЗ
  -> Добавлен товар: Помада (450 руб.)
  -> Способ доставки: Почта России. Уведомление отправлено на nastya2405@mail.com
  -> Добавлен товар: Тушь (200 руб.)
  -> Способ доставки: Почта России. Уведомление отправлено на nastya2405@mail.com
  Заказ №111 | Дата: 09.09.26 | Клиент: Анастасия | Оплата: Карта | Статус: Создан | Товаров: 2 | Сумма: 0 руб. | Изменен: 15:45:28
  [Онлайн] Тип: Онлайн-заказ | Трек: TRK-000 | Подарок: Нет
  Email: nastya2405@mail.com | Доставка: Почта России | Промокод: SALE2026 | Оплачен: Нет
  -> Итоговая сумма заказа №111: 650 руб.
  -> Статус заказа №111 изменен на 'Оплачен'
  -> Заказ №111 успешно оплачен через Карта
  -> Промокод 'SALE2026' применен к заказу №111
  -> [Явная реализация INotifiable] Email на nastya2405@mail.com: Ваш заказ собирается! (всего: 1)
  -> Статус заказа №111 изменен на 'Собирается'
  Заказ №111 | Дата: 09.09.26 | Клиент: Анастасия | Оплата: Карта | Статус: Собирается | Товаров: 2 | Сумма: 650 руб. | Изменен: 15:45:28
 